# TB2026-06 Commissioning High Gain Map

Quick check of converted commissioning ROOT files. The notebook reads `adc_high` from `siwecaldecoded`, drops the `-999` missing/unfilled sentinel written by `SLBdecoded2ROOT`, averages valid high gain ADC values for each `(chip, channel)`, maps them to the FEV11 COB xy geometry, and writes outputs next to the ROOT/dat file.

## Parameters

Change `RUN_NUMBER` for the normal TB2026 commissioning layout, or set `ROOT_FILE_PATH` directly to a converted ROOT file or run directory.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import uproot

DECODE_DIR = Path("/home/llr/ilc/shi/code/TB_2026_6/Decode")
DATA_ROOT = Path("/home/llr/ilc/shi/data/SiWECAL-Prototype/TB2026-06/Comission/tdc")
MAPPING_PATH = DECODE_DIR / "mapping" / "fev11_cob_rotate_chip_channel_x_y_mapping.txt"
OUTPUT_DIR = None  # None: write plots next to the ROOT/dat file.
ROOT_TORCH_PYTHON = Path("/data_ilc/flc/shi/miniconda3/envs/root_torch/bin/python")

RUN_NUMBER = 15
ROOT_FILE_PATH = None  # Example: DATA_ROOT / "ilc_run_000015" / "converted_ilc_run_000015.dat.root"
SLAB_INDEX = None  # None: use all slabs. Set an integer to inspect one slab.
INVALID_ADC = -999  # SLBdecoded2ROOT uses this for missing/unfilled bins; drop it in analysis.
USE_LOG_SCALE = False

print(f"Decode dir : {DECODE_DIR}")
print(f"Data root  : {DATA_ROOT}")
print(f"Mapping    : {MAPPING_PATH}")
print(f"Python     : {ROOT_TORCH_PYTHON}")

## Helpers

In [ ]:
def resolve_root_path(path, run_number: int) -> Path:
    if path is None:
        run_name = f"ilc_run_{run_number:06d}"
        return DATA_ROOT / run_name / f"converted_{run_name}.dat.root"
    path = Path(path)
    if path.is_dir():
        run_name = path.name
        return path / f"converted_{run_name}.dat.root"
    return path
def load_mapping(path: Path) -> np.ndarray:
    mapping = np.genfromtxt(path, names=True)
    required = {"chip", "channel", "x", "y"}
    if not required.issubset(mapping.dtype.names or ()):
        raise ValueError(f"Mapping file must contain columns: {sorted(required)}")
    return mapping
def resolve_output_dir(root_path: Path, output_dir) -> Path:
    if output_dir is None:
        return root_path.parent
    return Path(output_dir)


def read_fixed_adc_high(root_path: Path) -> tuple[np.ndarray, int]:
    if not root_path.exists():
        raise FileNotFoundError(
            f"Missing ROOT file: {root_path}\n"
            "Run: bash convert_tdc_runs.sh --run <RUN_NUMBER>"
        )

    with uproot.open(root_path) as root_file:
        tree = root_file["siwecaldecoded"]
        n_entries = tree.num_entries
        branch = tree["adc_high"]
        baskets = []
        for basket_index in range(branch.num_baskets):
            basket = branch.basket(basket_index)
            entry_start, entry_stop = branch.basket_entry_start_stop(basket_index)
            basket_entries = int(entry_stop - entry_start)
            data = np.frombuffer(basket.data, dtype=">i4")
            baskets.append(data.reshape(basket_entries, 15, 16, 15, 64).astype(np.int32))

    if not baskets:
        raise ValueError(f"No adc_high baskets in {root_path}")
    return np.concatenate(baskets, axis=0), n_entries


def load_adc_high(root_path: Path, slab_index: int | None = None) -> tuple[np.ndarray, np.ndarray, int, tuple[int, ...]]:
    values, n_entries = read_fixed_adc_high(root_path)
    raw_shape = values.shape
    valid_values = values[values != INVALID_ADC]

    values_for_mean = values.astype(float)
    values_for_mean[values_for_mean == INVALID_ADC] = np.nan
    if slab_index is None:
        chip_channel_mean = np.nanmean(values_for_mean, axis=(0, 1, 3))
    else:
        chip_channel_mean = np.nanmean(values_for_mean[:, slab_index, :, :, :], axis=(0, 2))

    return chip_channel_mean, valid_values, n_entries, raw_shape


def robust_limits(values: np.ndarray):
    finite = values[np.isfinite(values)]
    if finite.size == 0 or np.all(finite == finite[0]):
        return None, None
    return np.percentile(finite, 1), np.percentile(finite, 99)

## Load ROOT High Gain Means

In [ ]:
root_path = resolve_root_path(ROOT_FILE_PATH, RUN_NUMBER).resolve()
run_name = root_path.parent.name

output_dir = resolve_output_dir(root_path, OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

chip_channel_mean, adc_high_values, n_entries, adc_high_shape = load_adc_high(root_path, SLAB_INDEX)
mapping = load_mapping(MAPPING_PATH.resolve())

print(f"Input ROOT : {root_path}")
print(f"Output dir : {output_dir}")
print(f"Entries    : {n_entries}")
print(f"adc_high   : {adc_high_shape}")
print(f"Mean shape : {chip_channel_mean.shape}")
print(f"1D values  : {adc_high_values.size} after dropping {INVALID_ADC}")

## Build Maps

In [ ]:
N_XY_BINS = 32
N_CHIPS = 16
N_CHANNELS = 64

x_values = np.array(sorted(np.unique(mapping["x"])))
y_values = np.array(sorted(np.unique(mapping["y"])))
assert len(x_values) == N_XY_BINS, f"Expected {N_XY_BINS} x bins, got {len(x_values)}"
assert len(y_values) == N_XY_BINS, f"Expected {N_XY_BINS} y bins, got {len(y_values)}"

xy_hitmap = np.full((N_XY_BINS, N_XY_BINS), np.nan)

x_index = {value: i for i, value in enumerate(x_values)}
y_index = {value: i for i, value in enumerate(y_values)}

for row in mapping:
    chip = int(row["chip"])
    channel = int(row["channel"])
    xy_hitmap[y_index[row["y"]], x_index[row["x"]]] = chip_channel_mean[chip, channel]

chip_channel_hitmap = chip_channel_mean

print(f"XY hitmap bins          : {xy_hitmap.shape[0]} x {xy_hitmap.shape[1]}")
print(f"Chip/channel hitmap bins: {chip_channel_hitmap.shape[0]} x {chip_channel_hitmap.shape[1]}")

## High Gain 1D

In [ ]:
high_gain_values = adc_high_values

fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
ax.hist(
    high_gain_values,
    bins=80,
    histtype="stepfilled",
    color="tab:blue",
    alpha=0.75,
)
ax.set_title(f"{run_name} high gain distribution")
ax.set_xlabel("adc_high")
ax.set_ylabel("Counts")
ax.grid(alpha=0.25)

hist_output = output_dir / f"{run_name}_adc_high_hist1d.png"
fig.savefig(hist_output, dpi=160)
hist_output

## Geometry Hitmap

In [ ]:
plot_data = np.array(xy_hitmap, copy=True)
colorbar_label = "Mean adc_high"
suffix = "_log" if USE_LOG_SCALE else ""
if USE_LOG_SCALE:
    plot_data = np.log10(np.clip(plot_data, 1, None))
    colorbar_label = "log10(mean adc_high)"

x_bin_grid, y_bin_grid = np.meshgrid(np.arange(N_XY_BINS), np.arange(N_XY_BINS))
xy_edges = np.arange(-0.5, N_XY_BINS + 0.5, 1)
valid = np.isfinite(plot_data)
vmin, vmax = robust_limits(plot_data)
fig, ax = plt.subplots(figsize=(7.5, 6.4), constrained_layout=True)
_, _, _, image = ax.hist2d(
    x_bin_grid[valid].ravel(),
    y_bin_grid[valid].ravel(),
    bins=[xy_edges, xy_edges],
    weights=plot_data[valid].ravel(),
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
title = f"{run_name} adc_high mean"
ax.set_title(title)
ax.set_xlabel("x bin")
ax.set_ylabel("y bin")
ax.set_xticks(np.arange(0, N_XY_BINS, 4))
ax.set_yticks(np.arange(0, N_XY_BINS, 4))
ax.set_aspect("equal")
fig.colorbar(image, ax=ax, label=colorbar_label)

xy_output = output_dir / f"{run_name}_adc_high_mean_hitmap{suffix}.png"
fig.savefig(xy_output, dpi=160)
xy_output

## Chip/Channel View

In [ ]:
plot_data = np.array(chip_channel_hitmap, copy=True)
colorbar_label = "Mean adc_high"
suffix = "_log" if USE_LOG_SCALE else ""
if USE_LOG_SCALE:
    plot_data = np.log10(np.clip(plot_data, 1, None))
    colorbar_label = "log10(mean adc_high)"

channel_grid, chip_grid = np.meshgrid(np.arange(N_CHANNELS), np.arange(N_CHIPS))
channel_edges = np.arange(-0.5, N_CHANNELS + 0.5, 1)
chip_edges = np.arange(-0.5, N_CHIPS + 0.5, 1)
valid = np.isfinite(plot_data)
vmin, vmax = robust_limits(plot_data)
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
_, _, _, image = ax.hist2d(
    channel_grid[valid].ravel(),
    chip_grid[valid].ravel(),
    bins=[channel_edges, chip_edges],
    weights=plot_data[valid].ravel(),
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
ax.set_aspect("auto")
ax.set_title(f"{run_name} chip/channel adc_high mean")
ax.set_xlabel("channel bin")
ax.set_ylabel("chip bin")
ax.set_xticks(np.arange(0, N_CHANNELS, 8))
ax.set_yticks(np.arange(0, N_CHIPS, 2))
fig.colorbar(image, ax=ax, label=colorbar_label)

chip_output = output_dir / f"{run_name}_adc_high_mean_chip_channel{suffix}.png"
fig.savefig(chip_output, dpi=160)
chip_output